In [ ]:
# Setup for Google Colab
import os
import sys

if "google.colab" in sys.modules:
    !pip install -q mrcfile gemmi nglview ipywidgets==7.7.1 synth-core
    if not os.path.exists("/content/synth-cryo-em"):
        !git clone https://github.com/elkins/synth-cryo-em.git /content/synth-cryo-em
    sys.path.append("/content/synth-cryo-em/src")
    os.chdir("/content/synth-cryo-em")
    # Enable nglview for Colab
    from google.colab import output

    output.enable_custom_widget_manager()

# synth-cryo-em: Interactive Tutorial

Welcome to the interactive guide for `synth-cryo-em`. This notebook demonstrates how to generate synthetic Cryo-EM maps from atomic models and visualize the effects of various physical parameters.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from synth_core import add_gaussian_noise, compute_fsc

from synth_cryo_em.core import apply_ctf, generate_density_map

## 1. Create a Sample Structure
We'll start by creating a simple atomic model (a few carbon atoms in a row) to visualize.

In [ ]:
pdb_path = "tutorial_sample.pdb"
with open(pdb_path, "w") as f:
    f.write("""ATOM      1  CA  ALA A   1      10.000  10.000  10.000  1.00 20.00           C
ATOM      2  CA  ALA A   1      15.000  10.000  10.000  1.00 50.00           C
ATOM      3  CA  ALA A   1      20.000  10.000  10.000  1.00 100.00          C
TER
END
""")

## 2. Generate Basic Density
Let's see how the resolution parameter affects the "sharpness" of the atoms.

In [ ]:
res_high = 2.0
res_low = 8.0

grid_high, _ = generate_density_map(pdb_path, resolution=res_high)
grid_low, _ = generate_density_map(pdb_path, resolution=res_low)

data_high = np.array(grid_high, copy=False)
data_low = np.array(grid_low, copy=False)

# Visualize a slice through the center
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(data_high.sum(axis=0), cmap="viridis")
axes[0].set_title(f"High Resolution ({res_high}A)")
axes[1].imshow(data_low.sum(axis=0), cmap="viridis")
axes[1].set_title(f"Low Resolution ({res_low}A)")
plt.show()

## 3. Local Resolution (B-factors)
Now let's see how atomic B-factors (defined in our PDB as 20, 50, and 100) cause different atoms to blur independently.

In [ ]:
grid_b, _ = generate_density_map(pdb_path, resolution=2.0, use_bfactors=True)
data_b = np.array(grid_b, copy=False)

plt.figure(figsize=(8, 6))
plt.imshow(data_b.sum(axis=0), cmap="magma")
plt.title("Local Resolution via B-factors (20 -> 50 -> 100)")
plt.colorbar(label="Density")
plt.show()

## 4. CTF and Noise
Finally, we apply the Contrast Transfer Function and noise to see the "real-world" version.

In [ ]:
uc = grid_high.unit_cell
vox_size = (uc.a / grid_high.nu, uc.b / grid_high.nv, uc.c / grid_high.nw)

data_phys = apply_ctf(data_high, vox_size, defoc=1.5, b_factor=100)
data_noisy = add_gaussian_noise(data_phys, snr=2)

plt.figure(figsize=(8, 6))
plt.imshow(data_noisy.sum(axis=0), cmap="gray")
plt.title("Final Synthetic Map (CTF + Noise + B-factor Envelope)")
plt.show()

## 5. Quantitative Validation (FSC)
Let's compare our high-resolution ground truth with the noisy, physics-applied version using FSC.

In [ ]:
freqs, fsc = compute_fsc(data_high, data_noisy, vox_size)

plt.figure(figsize=(10, 6))
plt.plot(freqs, fsc, lw=2, color="blue")
plt.axhline(0.143, color="red", linestyle="--", label="0.143 Threshold")
plt.axhline(0.5, color="green", linestyle="--", label="0.5 Threshold")
plt.xlabel("Frequency (1/A)")
plt.ylabel("FSC")
plt.title("Fourier Shell Correlation: Ground Truth vs. Synthetic")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()